In [ ]:


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
NUM_SEEDS = 20
DIMS = [2, 5, 10, 20 ,30]
DIMS = [5, 10, 20 ,30,50]
DIMS = [10, 20 ,30,50]

AUTO_SCALING = True


METHODS = ['les_250_8',  'mes','logei','loghvarei', 'turbo', 'hci_gibo_09','sobol'] #'grad', 'nograd', 'gibo',
LABEL_NAMES = {'les_250_8':'LES (ours)',
'mes':'MES',
'turbo':'TuRBO',
'sobol':'Sobol random',
'logei':'logEI', 
'loghvarei':'logEI-DSP',
'hci_gibo':'HCI-GIBO',
'hci_gibo_09':'HCI-GIBO'}

In [24]:
def decompress_gibo(df):
    data = df[['y','n']].to_numpy(dtype=float)
    repeats = np.diff(data[:,-1].astype(int))
    repeats = np.insert(repeats, 0, data[0,-1])
    repeated = np.repeat(data[:, :-1], repeats, axis=0)
    return np.minimum.accumulate(repeated)

#### Comparison of performance

In [ ]:
from matplotlib.lines import Line2D

experiment_data = {}



within_mdl_experiments =  ['within_mdl_high/gpsample','within_mdl_medium/gpsample','within_mdl_low/gpsample','within_mdl_ext_low/gpsample']
oom_experiments = ['oom_high/gpsample','oom_medium/gpsample','oom_low/gpsample','oom_ext_low/gpsample']


for within_mdl_experiment,oom_experiment,name in zip(within_mdl_experiments,oom_experiments,["high","medium","low","ext_low"]):


    # load data

    for experiment in [oom_experiment,within_mdl_experiment]:

   
        BEST_HIST_PATH = f"./Data/{experiment}/optimizer_history/list_of_bests_"
        avrg_best_history = []
        std_best_history = []
        lower_quantiles_history = []
        upper_quantiles_history = []

        for dim in DIMS: 
            data_mean = []
            stds = []
            lower_quantile = []
            upper_quantile = []
            data_mean_normalized = []
            stds_normalized = []
            num_opjective_calls = min(20*dim, 400)
            for method in range(len(METHODS)):
                y_data = np.zeros((0,0))
                for seed in range(NUM_SEEDS):
                    file_identifier = f'{(seed+1):05d}_{dim}_{METHODS[method]}.csv'
                    try: 
                        table = pd.read_csv(BEST_HIST_PATH + file_identifier) 
                    except: 
                        print(f'Unable to find file {BEST_HIST_PATH+file_identifier}.')
                        continue
                    
                    if 'gibo' in METHODS[method]:
                        new_data = decompress_gibo(table.dropna())
                    else:
                        new_data = np.reshape(table['y'].to_numpy(), [-1,1])
                    if y_data.shape[0] == 0:
                        y_data = new_data[:num_opjective_calls, :]
                    else: 
                        new_data = new_data[:num_opjective_calls, :]
                        y_data = np.concatenate([y_data, new_data], axis=1)
                #data_mean.append(np.mean(y_data, axis=1))
                data_mean.append(np.median(y_data, axis=1))
                stds.append(np.std(y_data, axis=1))
                
                try:
                    lower_quantile.append(np.quantile(y_data, 0.25, axis=1))
                    upper_quantile.append(np.quantile(y_data, 0.75, axis=1))
                except:
                    lower_quantile.append([])
                    upper_quantile.append([])

            avrg_best_history.append(data_mean)
            #avrg_best_history_normalized.append(data_mean_normalized)
            std_best_history.append(stds)
            #std_best_history_normalized.append(stds_normalized)
            lower_quantiles_history.append(lower_quantile)
            upper_quantiles_history.append(upper_quantile)
        experiment_data[experiment] = {'avrg': avrg_best_history,
                                    'std' : std_best_history,
                                    'lower' : lower_quantiles_history,
                                    'upper' : upper_quantiles_history}
        
    
    import matplotlib as mpl
    from matplotlib.ticker import MaxNLocator
    from matplotlib.ticker import MultipleLocator

    # Change font to Computer Modern
    mpl.rcParams.update({
        "text.usetex": True,
        "font.family": "serif",  # LaTeX default is Computer Modern
        "font.serif": ["Computer Modern Roman"],
        "axes.unicode_minus": False  # to handle minus signs correctly
    })


    
    colors = anonymized


    width = 397.48 /72.27 # pt to in
    aspect_ratio = 0.45
    fontsize = 8
    linewidth = 0.8
    fig, axs = plt.subplots(2,len(DIMS), figsize=(width,width*aspect_ratio), sharey=True)

    #############################
    ### WITHIN MODEL          ###
    #############################
    avrg_best_history = experiment_data[within_mdl_experiment]['avrg']
    upper_quantiles_history = experiment_data[within_mdl_experiment]['upper']
    lower_quantiles_history = experiment_data[within_mdl_experiment]['lower']
    for dim, ax in enumerate(axs.flat[:len(DIMS)]):
        for method in range(len(METHODS)):
            try:
                ax.plot(avrg_best_history[dim][method], color=colors[method], linestyle='-', linewidth=linewidth, zorder=len(METHODS)-method+1)
                x = np.arange(1, std_best_history[dim][method].shape[0]+1)
                # ax.fill_between(x, avrg_best_history[dim][method] + std_best_history[dim][method], avrg_best_history[dim][method] - std_best_history[dim][method], 
                #                     color=colors[method], alpha=0.1)
                ax.fill_between(x, upper_quantiles_history[dim][method], lower_quantiles_history[dim][method], 
                                color=colors[method], alpha=0.2)
            except:
                print(f"Method number {method} not available")
        
        ax.set_xlabel('')
        ax.set_title(fr'$d = {DIMS[dim]}$', fontsize=fontsize)
        ax.set_xlim(0, len(avrg_best_history[dim][0]))
        ax.set_ylim(-11,0.6) 
        #ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=2))  # Adjust number of x-axis gridlines
        ax.xaxis.set_minor_locator(MaxNLocator(integer=True, nbins=4))  # Adjust number of y-axis gridlines
        ax.yaxis.set_minor_locator(MaxNLocator(integer=True, nbins=10))
        #ax.xaxis.set_minor_locator(MultipleLocator(50))  # or any consistent interval
        ax.set_xticks([0,np.ceil(len(avrg_best_history[dim][0])/100)*50,np.ceil(len(avrg_best_history[dim][0])/100)*100])
        ax.tick_params(axis='x', labelbottom=False)

        #ax.minorticks_on()
        ax.tick_params(axis='both', which='both', size=0)
        ax.grid(which='major', linestyle='-', linewidth=0.5, alpha=0.7) 
        ax.grid(which='minor', linestyle='--', linewidth=0.3, alpha=0.7) 
        ax.tick_params(axis='both', which='major', labelsize=fontsize)

        for spine in ax.spines.values():
            spine.set_zorder(100)

    #############################
    ### OUT OF MODEL          ###
    #############################
    avrg_best_history = experiment_data[oom_experiment]['avrg']
    upper_quantiles_history = experiment_data[oom_experiment]['upper']
    lower_quantiles_history = experiment_data[oom_experiment]['lower']
    for dim, ax in enumerate(axs.flat[len(DIMS):]):
        for method in range(len(METHODS)):
            ax.plot(avrg_best_history[dim][method], color=colors[method], linestyle='-', linewidth=linewidth, zorder=len(METHODS)-method+1)
            x = np.arange(1, upper_quantiles_history[dim][method].shape[0]+1)
            # ax.fill_between(x, avrg_best_history[dim][method] + std_best_history[dim][method], avrg_best_history[dim][method] - std_best_history[dim][method], 
            #                     color=colors[method], alpha=0.1)
            ax.fill_between(x, upper_quantiles_history[dim][method], lower_quantiles_history[dim][method], 
                                color=colors[method], alpha=0.2)
        
        ax.set_xlabel('')
        #ax.set_title(fr'$d = {DIMS[dim]}$', fontsize=fontsize)
        ax.set_xlim(0, len(avrg_best_history[dim][0]))
        ax.set_ylim(-11,0.6) 
        #ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=2))  # Adjust number of x-axis gridlines
        ax.xaxis.set_minor_locator(MaxNLocator(integer=True, nbins=4))  # Adjust number of y-axis gridlines
        ax.yaxis.set_minor_locator(MaxNLocator(integer=True, nbins=10))
        #ax.xaxis.set_minor_locator(MultipleLocator(50))  # or any consistent interval
        ax.set_xticks([0,np.ceil(len(avrg_best_history[dim][0])/100)*50,np.ceil(len(avrg_best_history[dim][0])/100)*100])

        #ax.minorticks_on()
        ax.tick_params(axis='both', which='both', size=0)
        ax.grid(which='major', linestyle='-', linewidth=0.5, alpha=0.7) 
        ax.grid(which='minor', linestyle='--', linewidth=0.3, alpha=0.7) 
        ax.tick_params(axis='both', which='major', labelsize=fontsize)

        for spine in ax.spines.values():
            spine.set_zorder(100)


    #############################
    ### GENERAL               ###
    #############################

    axs[0][0].set_yticks([x for x in range(-10,1,2)])
    #axs.flat[0].set_yticks(np.linspace(0, -2, 3))
    axs.flat[0].set_ylabel(r'$f(\hat{\mathbf{x}}^*)$', fontsize=fontsize)
    axs.flat[len(DIMS)].set_ylabel(r'$f(\hat{\mathbf{x}}^*)$', fontsize=fontsize)
    fig.text(0.5, 0.01, r'Evaluations \#', ha='center', fontsize=fontsize)
    legend_elements = [Line2D([0], [0], color=colors[i], lw=linewidth*1.2, linestyle='-', label=LABEL_NAMES[METHODS[i]]) for i in range(len(METHODS))]

    fig.text(0.5, 0.92, "Within Model Comparison", ha='center', va='center', fontsize=fontsize+1)
    fig.text(0.5, 0.45, "Out of Model Comparison", ha='center', va='center', fontsize=fontsize+1)

    fig.tight_layout(rect=[0, 0, 1, 0.93], h_pad=2.1)  # adjust layout to leave space below
    fig.legend(
        handles=legend_elements,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.2),#bbox_to_anchor=(0.5, -0.1),
        ncol=4,#len(METHODS),
        fontsize=fontsize,
        frameon=False,
    )
    plt.subplots_adjust(wspace=0.2)
    fig.savefig(name + "_both_comparisons.pdf", bbox_inches='tight', pad_inches=0.0)